In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [4]:
df = pd.read_csv("/content/db_drug_interactions.csv")

df.head()

,Drug 1,Drug 2,Interaction Description
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing a...
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosens...
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensiti...
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensiti...
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing ...


In [8]:
df['Interaction Description'][0]

'Trioxsalen may increase the photosensitizing activities of Verteporfin.'

In [5]:
df.columns

Index(['Drug 1', 'Drug 2', 'Interaction Description'], dtype='object')

## Class based prediction

In [6]:
# ===============================
# FULL PIPELINE: TF-IDF + SVM / RF
# Drug Interaction Prediction
# ===============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ===============================
# 1. Load Dataset
# ===============================
df = pd.read_csv("/content/db_drug_interactions.csv")

# ===============================
# 2. Combine Drug Inputs
# ===============================
df["text"] = df["Drug 1"].astype(str) + " " + df["Drug 2"].astype(str)

# ===============================
# 3. Create Labels (Rule-based)
# ===============================
def label_interaction(text):
    text = str(text).lower()

    if "photosensit" in text:
        return "photosensitivity"
    elif "increase" in text:
        return "increased_effect"
    elif "decrease" in text:
        return "reduced_effect"
    else:
        return "other"

df["label"] = df["Interaction Description"].apply(label_interaction)

# ===============================
# 4. TF-IDF Vectorization
# ===============================
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["text"])
y = df["label"]

# ===============================
# 5. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ===============================
# 6. Model Selection (choose ONE)
# ===============================

# OPTION A: SVM (Recommended for text)
model = LinearSVC()

# OPTION B: Random Forest (uncomment if needed)
# model = RandomForestClassifier(n_estimators=200, random_state=42)

# ===============================
# 7. Train Model
# ===============================
model.fit(X_train, y_train)

# ===============================
# 8. Prediction + Evaluation
# ===============================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

# ===============================
# 9. Prediction Function
# ===============================
def predict_interaction(drug1, drug2):
    text = drug1 + " " + drug2
    vec = tfidf.transform([text])
    return model.predict(vec)[0]

# ===============================
# 10. Example Test
# ===============================
print(predict_interaction("Trioxsalen", "Verteporfin"))

Accuracy: 0.8465895742514814
photosensitivity


## Full Description prediction

In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data
df = pd.read_csv("db_drug_interactions.csv")

# Combine input features
df["text"] = df["Drug 1"].astype(str) + " " + df["Drug 2"].astype(str)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = tfidf.fit_transform(df["text"])

# Function to predict full sentence
def predict_sentence(drug1, drug2):
    query = drug1 + " " + drug2
    query_vec = tfidf.transform([query])

    similarity = cosine_similarity(query_vec, X)
    idx = similarity.argmax()

    return df.iloc[idx]["Interaction Description"]

# Example
print(predict_sentence("Trioxsalen", "Verteporfin"))

Trioxsalen may increase the photosensitizing activities of Verteporfin.
